# 02 — BERT-alapú szentiment-címkézés / BERT-based sentiment labelling (GPU)

HU: Három, egymástól független megközelítés állít elő címkét (negatív / semleges / pozitív) és **1–10-es egész
pontszámot** a vélemény **szövegéből**:

| | Megközelítés / Approach | Tanítóadat / Training data | Mit tanítunk? / What do we train? |
|---|---|---|---|
| **A** | Zero-shot | — (kész modellek / ready models) | semmit / nothing |
| **B** | Transzfer / Transfer | HRAST (emberi címke / human labels) | BERT finomhangolás / fine-tuning |
| **C** | Önképzés / Self-training | Booking + A/B legbiztosabb címkéi / most confident labels | BERT finomhangolás / fine-tuning |

**Futtatás / How to run:** Google Colab, T4 GPU (*Runtime → Change runtime type → T4 GPU*), a teljes `CODE`
mappa feltöltve a Google Drive-ra. Saját 4 GB-os GPU-n is fut (kisebb batch, lásd `config.py`).
Minden előrejelzés darabonként mentődik → **egy megszakadt futás folytatható**.

| Lépés / Step | T4, 20k vélemény / reviews (becslés / estimate) |
|---|---|
| A (2 modell / models) | ~3–5 min |
| B tanítás + címkézés / training + labelling | ~15 + 5 min |
| C tanítás + címkézés / training + labelling | ~15 + 2 min |

## 1. Környezet / Environment

In [2]:
# HU: Colab-on: Drive csatolása, belépés a projektmappába, csomagok telepítése.
#     A PROJECT_DIR-t a saját Drive-útvonaladra állítsd!
# EN: On Colab: mount Drive, enter the project folder, install packages.
#     Set PROJECT_DIR to your own Drive path!
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/THESIS/CODE"
    os.chdir(os.path.join(PROJECT_DIR, "notebooks"))
    get_ipython().system("pip install -q transformers accelerate pyarrow")
print("Working directory:", os.getcwd())

Working directory: c:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\notebooks


In [3]:
# HU: A notebooks/ mappából futunk, ezért a szülőmappát a path-hoz adjuk (src csomag).
# EN: We run from the notebooks/ folder, so the parent folder is added to the path (src package).
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import set_seed
from src.config import CFG, PATHS, SENTIMENT_COLORS, INK, APPROACHES, apply_plot_style
from src import data_loading as dl
from src import preprocessing as pp
from src import evaluation as ev
from src import bert_labelling as bl
from src import bert_training as bt
from src import label_validation as lv
from src import pipeline

# HU: Reprodukálhatóság + egységes ábrastílus. / EN: Reproducibility + uniform figure style.
set_seed()
PATHS.ensure()
apply_plot_style()
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 170)

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("!!! NO GPU — labelling will be very slow on a CPU !!!")

PyTorch 2.14.0+cu132 | CUDA: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU | VRAM: 4.3 GB


In [4]:
# HU: A 01-es notebook által mentett munkahalmaz. / EN: The working set saved by notebook 01.
working = dl.read_frame(pipeline.WORKING_SET)
print(f"Working set: {len(working):,} reviews")

Working set: 20,000 reviews


In [4]:
# HU: Valódi token-hossz a 3 osztályos modell tokenizálójával (2000-es mintán).
# EN: Real token length with the 3-class model's tokenizer (on a sample of 2000).
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(CFG.zero_shot_3class_model)
sample_texts = working["text_bert"].sample(min(2000, len(working)), random_state=CFG.random_seed)
lens = np.array([len(tok(t)["input_ids"]) for t in sample_texts])
print(f"Tokens — median {np.median(lens):.0f}, 95%: {np.percentile(lens, 95):.0f}, max {lens.max()}")
print(f"{(lens > CFG.max_length).mean():.2%} of the reviews exceed max_length={CFG.max_length}.")

10:53:12 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
10:53:12 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
10:53:12 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
10:53:12 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cardiffnlp/twitter-roberta-base-sentiment-latest/3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7/config.json?%2Fcardiffnlp%2Ftwitter-roberta-base-sentiment-latest%2Fresolve%2Fmain%2Fconfig.json=&etag=%22baae943f01ad3371bc9d543de3229512dfe27cb8%22 "HTTP/1.1 200 OK"
10:53:12 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/cardiffnlp/twitter-roberta-base-sentiment-latest/3216a57f2a0d9c45a2e6c20157

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

c:\Users\seszt\Documents\BCE\7_felev\Szakdolgozat\THESIS\CODE\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\seszt\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment-latest. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
10:53:12 | INFO    | httpx | HTTP Request: HEAD http

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

10:53:13 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/merges.txt "HTTP/1.1 307 Temporary Redirect"
10:53:13 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cardiffnlp/twitter-roberta-base-sentiment-latest/3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7/merges.txt?%2Fcardiffnlp%2Ftwitter-roberta-base-sentiment-latest%2Fresolve%2Fmain%2Fmerges.txt=&etag=%22226b0752cac7789c48f0cb3ec53eda48b7be36cc%22 "HTTP/1.1 200 OK"
10:53:13 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/cardiffnlp/twitter-roberta-base-sentiment-latest/3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7/merges.txt?%2Fcardiffnlp%2Ftwitter-roberta-base-sentiment-latest%2Fresolve%2Fmain%2Fmerges.txt=&etag=%22226b0752cac7789c48f0cb3ec53eda48b7be36cc%22 "HTTP/1.1 200 OK"


merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

10:53:13 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/tokenizer.json "HTTP/1.1 404 Not Found"
10:53:13 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
10:53:14 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
10:53:14 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cardiffnlp/twitter-roberta-base-sentiment-latest/3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7/special_tokens_map.json?%2Fcardiffnlp%2Ftwitter-roberta-base-sentiment-latest%2Fresolve%2Fmain%2Fspecial_tokens_map.json=&etag=%222ea7ad0e45a9d1d1591782ba7e29a703d0758831%22 "HTTP/1.1 200 OK"
10:53:14 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

10:53:14 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Tokens — median 49, 95%: 147, max 469
0.75% of the reviews exceed max_length=256.


## 2. (A) Zero-shot címkézés / Zero-shot labelling

HU: Két, mások által már finomhangolt modell — mi nem tanítunk semmit:
* **3 osztályos modell** (`CFG.zero_shot_3class_model`) → `A_label` (negatív / semleges / pozitív);
* **1–5 csillagos BERT-modell** (`CFG.zero_shot_stars_model`) → `A_score10`:
  várható csillagszám $E=\sum_k k\,P(k)$, majd $\text{score}=1+9\,(E-1)/4$, egészre kerekítve.

Korlát: egyik modell sem hotelvéleményeken tanult (tweetek, termékvélemények) → doménkülönbség.

EN: Two models already fine-tuned by others — we train nothing: a **3-class model** → `A_label`; a **1–5 star BERT
model** → `A_score10` (expected stars rescaled to 1–10). Limitation: neither was trained on hotel reviews → domain gap.

In [5]:
scorers_a = {"3class": bl.TransformerScorer(CFG.zero_shot_3class_model),
             "stars": bl.TransformerScorer(CFG.zero_shot_stars_model)}
labels_a = bl.label_approach_a(working, scorers=scorers_a)
bl.save_labels(labels_a, "labels_A")

display(labels_a["A_label"].value_counts(normalize=True).round(3).to_frame("share"))
display(labels_a[["A_score10", "A_score10_polarity", "A_expected_stars"]].describe().round(2))

10:56:26 | INFO    | src.bert_labelling | Loading model: cardiffnlp/twitter-roberta-base-sentiment-latest (device: cuda, fp16: True)
10:56:26 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
10:56:26 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cardiffnlp/twitter-roberta-base-sentiment-latest/3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7/config.json?%2Fcardiffnlp%2Ftwitter-roberta-base-sentiment-latest%2Fresolve%2Fmain%2Fconfig.json=&etag=%22baae943f01ad3371bc9d543de3229512dfe27cb8%22 "HTTP/1.1 200 OK"
10:56:26 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
10:56:26 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/tokenizer_config.json

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

10:56:45 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
10:56:45 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-roberta-base-sentiment-latest "HTTP/1.1 200 OK"
10:56:46 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-roberta-base-sentiment-latest/commits/main "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

10:56:46 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-roberta-base-sentiment-latest/discussions?p=0 "HTTP/1.1 200 OK"
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
10:56:46 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-roberta-base-sentiment-latest/commits/refs%2Fpr%2F43 "HTTP/1.1 200 OK"
10:56:46 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/refs%2Fpr%2F43/model.safetensors.index.json "HTTP/1.1 404 Not Found"
10:56:46 | INFO    | httpx | HTTP Request: H

... (13 outputs left out)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

10:57:08 | INFO    | src.bert_labelling | Model type: stars | labels: [1, 2, 3, 4, 5]
10:57:08 | INFO    | src.bert_labelling | [A_3class_review] chunk 1/2 (10000 texts)
10:57:13 | INFO    | src.bert_labelling |   twitter-roberta-base-sentiment-latest: 50/157 batches (731 texts/s)
10:57:18 | INFO    | src.bert_labelling |   twitter-roberta-base-sentiment-latest: 100/157 batches (650 texts/s)
10:57:28 | INFO    | src.bert_labelling |   twitter-roberta-base-sentiment-latest: 150/157 batches (476 texts/s)
10:57:31 | INFO    | src.bert_labelling | [A_3class_review] chunk 2/2 (10000 texts)
10:57:34 | INFO    | src.bert_labelling |   twitter-roberta-base-sentiment-latest: 50/157 batches (988 texts/s)
10:57:40 | INFO    | src.bert_labelling |   twitter-roberta-base-sentiment-latest: 100/157 batches (724 texts/s)
10:57:51 | INFO    | src.bert_labelling |   twitter-roberta-base-sentiment-latest: 150/157 batches (495 texts/s)
10:57:53 | INFO    | src.bert_labelling | [A_stars_review] chunk 1/2 (

,share
A_label,
positive,0.928
neutral,0.038
negative,0.034


,A_score10,A_score10_polarity,A_expected_stars
count,20000.00,20000.00,20000.00
mean,8.19,9.35,4.19
std,1.52,1.59,0.66
min,1.00,1.00,1.12
25%,7.00,10.00,3.89
50%,9.00,10.00,4.35
75%,9.00,10.00,4.69
max,10.00,10.00,4.99


In [6]:
# HU: Belső egyezés: a 3 osztályos és a csillagos modell mennyire ért egyet?
# EN: Internal agreement: how far do the 3-class and the star model agree?
tab = lv.approach_crosstab(labels_a, "A_label", "A_star_label")
display(tab)
print(f"Cohen's kappa (3-class vs stars): {lv.cohen_kappa(labels_a['A_label'], labels_a['A_star_label']):.3f}")
ev.save_table(tab, "02_A_internal_agreement")

A_star_label,negative,neutral,positive
A_label,,,
negative,0.4296,0.4707,0.0997
neutral,0.0892,0.4423,0.4685
positive,0.0097,0.0906,0.8997


10:59:23 | INFO    | src.evaluation | Table saved: 02_A_internal_agreement.csv


Cohen's kappa (3-class vs stars): 0.333


WindowsPath('C:/Users/seszt/Documents/BCE/7_felev/Szakdolgozat/THESIS/CODE/results/tables/02_A_internal_agreement.csv')

## 3. (B) Transzfertanulás a HRAST-on / Transfer learning on HRAST

HU: A HRAST kézzel annotált, mondatszintű hotel-adat. A `bert-base-uncased` modellt ezen finomhangoljuk, majd a
Booking-véleményeket **mondatokra bontva** címkézzük, és a mondatok valószínűségét átlagoljuk.
A HRAST teszthalmazán mért eredmény a projekt egyetlen **emberi címkén** mért teljesítménye (a gold mintán kívül).
Tedd a HRAST CSV-t a `data/raw/hrast/` mappába. A konzulenssel egyeztetni kell, mert korábban a HRAST kimaradt —
itt **csak a címkéző tanítóadata**, nem elemzett adat.

EN: HRAST is human-annotated, sentence-level hotel data. `bert-base-uncased` is fine-tuned on it, then the Booking
reviews are labelled **sentence by sentence** and the sentence probabilities are averaged. The HRAST test result is
the project's only performance measured on **human labels** (besides the gold sample). Put the HRAST CSV into
`data/raw/hrast/`. To be agreed with the supervisor: HRAST is used **only as training data of the labeller**.

In [5]:
HRAST_AVAILABLE = dl.find_hrast_file() is not None
print("HRAST available:", HRAST_AVAILABLE)
if HRAST_AVAILABLE:
    hrast_raw = dl.load_hrast_raw()
    display(hrast_raw.head())
    # HU: Ha az oszlopokat nem ismeri fel: prepare_hrast_sentences(hrast_raw, text_col="...", sentiment_col="...")
    # EN: If columns are not recognised: prepare_hrast_sentences(hrast_raw, text_col="...", sentiment_col="...")
    hrast = dl.prepare_hrast_sentences(hrast_raw)
    display(hrast["label"].value_counts().to_frame("sentences"))

12:02:05 | INFO    | src.data_loading | HRAST loaded: hrast.csv — 23113 rows, columns: ['id', 'review', 'Unnamed: 2', 'positive', 'negative', 'neutral', 'Clean', 'Comfort', 'Facilities/Amenities', 'Location', 'Restaurant (dinner)', 'Staff', 'View (Balcony)', 'Breakfast', 'Room', 'Pool', 'Beach', 'Bathroom/Shower (toilet)', 'Bar', 'Bed', 'Parking', 'Noise', 'Reception-checkin', 'Lift', 'Value for money', 'Wi-Fi', 'Generic', 'Aspect']


HRAST available: True


,id,review,Unnamed: 2,positive,negative,neutral,Clean,Comfort,Facilities/Amenities,Location,...,Bar,Bed,Parking,Noise,Reception-checkin,Lift,Value for money,Wi-Fi,Generic,Aspect
0,1,1 elevator not working.,NaN,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,2,1 euros to park per day on their own premises.,NaN,0,0,1,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
2,3,1 hour queue to check-in.,NaN,0,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,4,1 min walk to metro entrance and also few min walk to the central station where you can access rail trains.,NaN,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,5,"1 minute walk from the main metro & train station, which made getting around Copenhagen extremely easy.",NaN,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


12:02:05 | INFO    | src.data_loading | HRAST columns used — text: review | sentiment: one-hot ['negative', 'neutral', 'positive'] | group: None
12:02:05 | INFO    | src.data_loading | HRAST one-hot rows — positive: 11818 | negative: 10503 | neutral only: 791 | both pos+neg: 1 | none: 0
12:02:07 | INFO    | src.data_loading | HRAST sentences: 23104 (dropped as conflicting: 1) | distribution: {'positive': 11812, 'negative': 10501, 'neutral': 791}


,sentences
label,
positive,11812
negative,10501
neutral,791


In [ ]:
if HRAST_AVAILABLE:
    result_b = bt.train_approach_b(hrast)
    hrast_test = pd.DataFrame([result_b["test_metrics"]], index=["HRAST test"]).round(4)
    display(hrast_test)
    ev.save_table(hrast_test, "02_approach_b_hrast_test")
    ev.plot_confusion_matrix(result_b["test_predictions"]["label"], result_b["test_predictions"]["predicted"],
                             title="(B) on HRAST test", save_as="02_approach_b_hrast_confusion")
    plt.show()

In [ ]:
if HRAST_AVAILABLE:
    scorer_b = bl.TransformerScorer(str(PATHS.models / "approach_b_hrast"))
    labels_b = bl.label_approach_b(working, scorer=scorer_b)
    bl.save_labels(labels_b, "labels_B")
    display(labels_b["B_label"].value_counts(normalize=True).round(3).to_frame("share"))
    print(f"Mixed reviews (a confidently positive AND negative sentence): {labels_b['B_is_mixed'].mean():.1%}")

    # HU: Mondatszintű előrejelzések megtartása az aspektus-elemzéshez (a cache miatt nincs újraszámolás).
    # EN: Keep sentence-level predictions for the aspect analysis (no recomputation thanks to the cache).
    sentences_b = bl.predict_sentences(working, "", "B", cache_name="B_sentence", scorer=scorer_b)
    dl.write_frame(sentences_b, PATHS.labels / "sentences_B")

## 4. (C) Önképzés a Booking-adaton / Self-training on the Booking data

HU: A tanár(ok) — (A) és ha van, (B) — **legbiztosabb és egybehangzó** címkéi alkotják a tanítóhalmazt
(`CFG.pseudo_label_min_confidence`, `CFG.pseudo_label_require_agreement`). Erre finomhangolunk egy BERT-et, amely
így a **saját doménen** (Booking-szöveg) tanul, és a bizonytalan véleményeket is címkézi.
**Figyelem:** a (C) validációs eredménye a tanárokkal való egyezést méri, nem a valódi pontosságot.

EN: The **most confident and agreeing** labels of the teacher(s) — (A) and, if available, (B) — form the training
set. A BERT fine-tuned on it learns in the **own domain** (Booking text) and also labels the uncertain reviews.
**Warning:** the validation result of (C) measures agreement with the teachers, not true accuracy.

In [ ]:
teachers = labels_a.join(labels_b, how="inner") if HRAST_AVAILABLE else labels_a
pseudo = bl.select_pseudo_labels(teachers)
display(pseudo["pseudo_label"].value_counts().to_frame("training examples"))

result_c = bt.train_approach_c(working, pseudo)
print("Validation (agreement with the teachers):", {k: round(v, 4) for k, v in result_c["val_metrics"].items()
                                                    if k in ("accuracy", "f1_macro")})

In [ ]:
scorer_c = bl.TransformerScorer(str(PATHS.models / "approach_c_self_training"))
labels_c = bl.label_approach_c(working, scorer=scorer_c)
bl.save_labels(labels_c, "labels_C")
display(labels_c["C_label"].value_counts(normalize=True).round(3).to_frame("share"))
print(f"Reviews (C) was trained on (excluded from comparisons): {labels_c['C_in_training'].mean():.1%}")

## 5. Booking-mezőpróba / Booking field test

HU: A `review_positive` mező szövegének pozitívnak, a `review_negative` mezőének negatívnak kell lennie — a vendég
maga döntötte el, hova ír. Ez címke nélküli, mégis **független** ellenőrzés mindhárom modellre.
EN: Text of the `review_positive` field should be positive, of `review_negative` negative — the guest chose the field.
An **independent** check of all three models without labels.

In [ ]:
field_set = lv.build_field_test_set(working)
field_scorers = {"A": scorers_a["3class"], "C": scorer_c}
if HRAST_AVAILABLE:
    field_scorers["B"] = scorer_b

field_preds = field_set.copy()
reports = []
for name, scorer in field_scorers.items():
    probs = scorer.predict_proba(field_set["text_bert"].tolist())
    field_preds[f"{name}_label"] = np.array(bl.LABELS)[probs.argmax(axis=1)]
    reports.append(lv.field_test_report(field_set["expected"], field_preds[f"{name}_label"]).assign(approach=name))

field_report = pd.concat(reports).reset_index().set_index(["approach", "field"])
display(field_report)
ev.save_table(field_report, "02_field_test")
dl.write_frame(field_preds, PATHS.labels / "field_test_predictions")

## 6. Végső címke / Final label

HU: Többségi szavazat a három megközelítés között (döntetlennél `CFG.tie_breaker_approach`), a végső 1–10-es pontszám
a folytonos pontszámok átlaga. Az `unanimous` oszlop jelöli a legbiztosabb címkéket.
EN: Majority vote of the three approaches (tie: `CFG.tie_breaker_approach`); the final 1–10 score is the mean of the
continuous scores. The `unanimous` column flags the most reliable labels.

In [ ]:
labelled = pipeline.step_combine()
display(labelled["label_final"].value_counts(normalize=True).round(3).to_frame("share"))
print(f"Unanimous: {labelled['unanimous'].mean():.1%} of the reviews")
display(labelled[["text", "score", "label_final", "score10_final", "n_agree"]].head(8))
print("\nNext: 03_label_validation.ipynb (runs on a CPU, locally)")